# 5. Classify, shortlist and score

Load the best classifier and best high-popularity regressor. Both receive original features and apply their own saved preprocessing. This replaces the original missing-model path and incompatible PCA hand-off. The classifier is reused instead of running another 1,500-candidate search.

In [1]:
from pipeline_lib import RunContext, run_combined, write_report
ctx = RunContext.from_environment("combined")
print("Profile:", ctx.config["profile"])
print("Artifacts:", ctx.run_dir)
from IPython.display import Markdown, display

Profile: full
Artifacts: /Users/anishech/Projects/ML_Spotify/runs/20261021_141923_142263_full


Use the shared holdout only. Keep tracks predicted as hits, score their popularity and sort by predicted score. Track names and artists are joined back for interpretation. Shortlist regression includes false positives; it does not measure performance on all songs.

In [2]:
ranked_candidates = run_combined(ctx)
display(ranked_candidates)

Combined holdout: 651 shortlisted / 14389; metrics {'rows': 14389, 'actual_hits': 456, 'predicted_hits': 651, 'accuracy': 0.9462784071165473, 'balanced_accuracy': 0.6657451985126817, 'hit_precision': 0.2565284178187404, 'hit_recall': 0.36622807017543857, 'hit_f1': 0.3017163504968383, 'confusion_matrix': [[13449, 484], [289, 167]], 'average_precision': 0.212920892744139, 'roc_auc': 0.8480424330221952, 'shortlist_regression': {'rows': 651, 'mae': 30.89337694783609, 'rmse': 42.65376485401698, 'r2': -0.9903270662210535, 'within_5_percent': 19.969278033794165}, 'shortlist_mean_baseline': {'rows': 651, 'mae': 30.560177598215606, 'rmse': 41.94339841396969, 'r2': -0.9245843044336837, 'within_5_percent': 16.129032258064516}}


,row_id,actual_popularity,predicted_popularity,hit_score,track_id,artists,album_name,track_name
0,17776,14,81.681416,0.677530,3Xgc0P2nr8h4qkCN8eiK25,Five Star Hotel,Machine Girl / Five Star Hotel,Blister
1,20727,0,80.973292,0.999990,3ym8ZuRQtZ7rSEbXkT3eJE,Ty Dolla $ign;Jhené Aiko;Mustard,20's R'n'B,By Yourself
2,81668,84,80.782494,0.996945,1XGmzt0PVuFgQYYnV2It7A,Maroon 5;Wiz Khalifa,Overexposed Track By Track,Payphone
3,68951,83,80.742704,0.998744,35wvL50xvKpCHEJPxLOLPI,Bad Bunny,X 100PRE,Si Estuviésemos Juntos
4,68692,49,80.694196,0.999015,1qkVJlyVaLZbwolRZmc17i,Kevin Martes 13;Pablo Chill-E,Perco Con Mari,Perco Con Mari
5,20600,83,80.627987,0.999997,6XO8RlYuJCiI0v3IA48FeJ,Marshmello;Juice WRLD,Bye Bye,Bye Bye
6,81882,73,80.617008,0.985800,4TgxFMOn5yoESW6zCidCXL,Jawsh 685;Jason Derulo;BTS,Savage Love (Laxed - Siren Beat) [BTS Remix],Savage Love (Laxed – Siren Beat) [BTS Remix]
7,81146,70,80.592755,0.997922,6PO6EnP2QoP4vN27KmKvrj,Prem Dhillon,No Lookin' Back,Nah They Can’t
8,81207,86,80.582772,0.975370,4iJyoBOLtHqaGxP12qzhQI,Justin Bieber;Daniel Caesar;Giveon,Justice,Peaches (feat. Daniel Caesar & Giveon)
9,20669,0,80.563995,0.999905,2puCdLnS6sKRdbEabw4bjP,Wiz Khalifa,Ghetto Superstar - New Rap Classics,Bad Ass Bitches


In [3]:
report = write_report(ctx)
display(Markdown(report.read_text()))

# Spotify results: full

This run predicts Spotify popularity, not measured Billboard chart membership.

Raw rows: 114,000; cleaned: 71,945; modeled: 71,945.
Training rows: 57,556; holdout rows: 14,389 (456 actual hits). Hit means popularity > 70.

All stages share a saved stratified holdout. Every estimator stores its own genre encoder, scaler, PCA and model. Preprocessing is fitted within training folds; genre training encodings are cross-fitted.
Winners are selected using cross-validation, never holdout scores. Training metrics use the final refitted encoder and are descriptive only.

## Measured results

### classification: GaussianNB

Accuracy 94.63%; majority baseline 96.83%.
Hit precision 25.65%; recall 36.62%; F1 0.3017; average precision 0.2129.
Confusion matrix [[TN, FP], [FN, TP]]: [[13449, 484], [289, 167]].
Selection metric: hit_f1; CV score 0.3257.

### regression: XGBRegressor

MAE 11.082; RMSE 15.611; R² 0.367; within ±5 points 37.10%.
Mean-prediction baseline: MAE 16.274; RMSE 19.615; R² -0.000.
Selection metric: neg_rmse; CV score -15.6191.

### regression_high: ElasticNet

MAE 3.268; RMSE 4.050; R² 0.242; within ±5 points 82.24%.
Mean-prediction baseline: MAE 3.873; RMSE 4.655; R² -0.002.
Selection metric: neg_rmse; CV score -4.4462.

## Combined workflow

Shortlisted 651 tracks from 14389 holdout tracks.
The high-popularity regressor scores all shortlisted tracks, including false positives. These metrics describe the shortlist, not every track or future Billboard success.
Shortlist regression: {"rows": 651, "mae": 30.89337694783609, "rmse": 42.65376485401698, "r2": -0.9903270662210535, "within_5_percent": 19.969278033794165}.

## Interpretation and limits

A majority-class baseline can have high accuracy while finding zero hits. Use hit precision, recall, F1 and average precision to judge discovery. Regression R² must be compared to a mean baseline. Within ±5 points is a tolerance metric, not classification accuracy.

The original saved RF classifier had 96.70% accuracy on a different random split. The original general-regression Lasso had RMSE 15.542 and R² 0.371; the high-popularity Lasso had RMSE 4.179 and R² 0.223. These are historical notebook outputs, not rerun results. Different samples, stratification, cross-fitted encoding, valid validation and selection criteria mean this run is not a direct reproduction of those numbers.

Random track splitting does not test future releases or independence between artists. Song characteristics alone omit marketing, audience and release timing. The high-popularity model is trained on known hits and may overestimate false positives.

Open `metrics/*_summary.csv` for all models, `predictions/ranked_candidates.csv` for the shortlist, and `executed_notebooks/` for cell-by-cell outputs. Timing and hashes are recorded in `run_manifest.json`.


Artifacts include per-model metrics, CV tables, predictions, complete model pipelines, plots and executed notebooks. The run manifest records configuration, timing, interpreter, input and source hashes. Read `RESULTS.md` for this run; a smoke test is a workflow check, not the final model comparison.